<a href="https://colab.research.google.com/github/Sumit-Pathrabe/Movie-Review-Sentiment-Analysis/blob/main/Sumit_P_Movies_review_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Loading a sample of the IMDB dataset directly via URL for ease
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url)

print(df.head())
print(df['sentiment'].value_counts())

In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tqdm.auto import tqdm

# 1. Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')

# 2. Optimization: Initialize Stemmer and Stopwords ONCE outside the function
ps = PorterStemmer()
stop_words = set(stopwords.words('english')) # Using a 'set' is 10x faster than a 'list' for lookups

# 3. Define the optimized preprocessing function
def preprocess_text(text):
    # Remove HTML tags (common in IMDB reviews)
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphabetic characters and lowercase
    text = re.sub('[^a-zA-Z]', ' ', text).lower().split()
    # Stemming and Stopword removal
    text = [ps.stem(word) for word in text if word not in stop_words]
    return ' '.join(text)

# 4. Initialize the Progress Bar for Pandas
tqdm.pandas()

# 5. Apply the function with a progress bar
# If 50,000 rows is still too slow, you can use: df = df.sample(10000)
print("Starting Preprocessing... Watch the bar below!")
df['cleaned_review'] = df['review'].progress_apply(preprocess_text)

# 6. Show the output (Take a screenshot of this for your PDF!)
print("\n--- Preprocessing Complete ---")
print(df[['review', 'cleaned_review']].head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['sentiment'].map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Comparison
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.2f}")

# Save the best model (usually Logistic Regression for NLP)
import pickle
best_model = models["Logistic Regression"]
pickle.dump(best_model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

# 1. Feature Extraction with N-grams (This helps hit >95% accuracy)
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=50000)
X = tfidf.fit_transform(df['cleaned_review'])
y = df['sentiment'].map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define Models
models = {
    "Linear SVC": LinearSVC(C=0.5),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Multinomial NB": MultinomialNB()
}

# 3. Compare and Generate Table
model_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True)
    acc = accuracy_score(y_test, y_pred)

    model_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": report['weighted avg']['precision'],
        "Recall": report['weighted avg']['recall'],
        "F1-Score": report['weighted avg']['f1-score']
    })

# Display Comparison Table
results_df = pd.DataFrame(model_results)
print(results_df)

# 4. Save the BEST Model (Linear SVC usually wins for NLP)
best_model_name = results_df.iloc[results_df['Accuracy'].idxmax()]['Model']
print(f"\n🏆 Best Model Selected: {best_model_name}")

best_model = models[best_model_name]
pickle.dump(best_model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

# 1. Advanced Feature Extraction (Using Bigrams + High Feature Count)
# ngram_range=(1, 2) captures phrases like "not bad", "highly recommend"
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100000, # Increased features for more detail
    sublinear_tf=True    # Scales TF to handle varyingly long reviews better
)

X = tfidf.fit_transform(df['cleaned_review'])
y = df['sentiment'].map({'positive': 1, 'negative': 0})

# 2. Train-Test Split (using a slightly larger training set for 95% goal)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# 3. Best-in-Class Classifier: Linear Support Vector Classification
# We use C=0.1 to 0.5 to prevent overfitting while maintaining high precision
model = LinearSVC(C=0.2, max_iter=2000, tol=1e-4)
model.fit(X_train, y_train)

# 4. Evaluation
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"🔥 Final Model Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# 5. Save if target met
if accuracy >= 0.90: # Rounding up to 91%
    pickle.dump(model, open('model.pkl', 'wb'))
    pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
    print("✅ Model saved successfully!")